# 🦋 LensArthropoda — Smart Insect Identifier
## Notebook Pelatihan Model

| Item | Detail |
|---|---|
| **Dataset** | Insects (baxtiyorbotiraliyev/insects) |
| **Arsitektur** | EfficientNet-B3 (Transfer Learning) |
| **Framework** | PyTorch + timm |
| **Platform** | Kaggle GPU / Google Colab |

---

## 📦 1. Install Dependencies

In [ ]:
# Jalankan sekali jika library belum tersedia
# !pip install timm scikit-learn matplotlib seaborn Pillow tqdm --quiet

## 📚 2. Import Library

In [ ]:
import os
import json
import random
import warnings
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from PIL import Image, UnidentifiedImageError

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import transforms, datasets
import timm

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
)
from tqdm import tqdm

warnings.filterwarnings("ignore")

# ── Reproducibility ──────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device  : {DEVICE}")
print(f"PyTorch : {torch.__version__}")
if torch.cuda.is_available():
    print(f"GPU     : {torch.cuda.get_device_name(0)}")
    print(f"VRAM    : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

## ⚙️ 3. Konfigurasi & Path Dataset

> **Struktur dataset (baxtiyorbotiraliyev/insects):**
> ```
> /kaggle/input/datasets/baxtiyorbotiraliyev/insects/
> └── dataset/
>     ├── train/   ← folder kelas serangga
>     ├── test/    ← folder kelas serangga
>     ├── val/     ← folder kelas serangga
>     ├── classes.txt
>     ├── train.txt
>     ├── test.txt
>     └── val.txt
> ```

In [ ]:
# ============================================================
# PATH DATASET — sesuaikan dengan platform yang digunakan
# ============================================================

# Kaggle (baxtiyorbotiraliyev/insects)
KAGGLE_ROOT    = "/kaggle/input/datasets/baxtiyorbotiraliyev/insects/dataset"

# Google Colab (jika menggunakan Google Drive)
COLAB_ROOT     = "/content/drive/MyDrive/insects/dataset"

# ── Auto-detect platform ─────────────────────────────────────
if os.path.isdir(KAGGLE_ROOT):
    ROOT = KAGGLE_ROOT
    print("✅ Platform : Kaggle")
elif os.path.isdir(COLAB_ROOT):
    ROOT = COLAB_ROOT
    print("✅ Platform : Google Colab")
else:
    # Fallback manual — ubah path di sini jika berbeda
    ROOT = KAGGLE_ROOT
    print(f"⚠️  Path tidak ditemukan otomatis, menggunakan: {ROOT}")

TRAIN_DIR = os.path.join(ROOT, "train")
TEST_DIR  = os.path.join(ROOT, "test")
VAL_DIR   = os.path.join(ROOT, "val")

# Hyperparameter training
IMG_SIZE     = 224
BATCH_SIZE   = 32
NUM_EPOCHS   = 30
LR           = 3e-4
WEIGHT_DECAY = 1e-4
PATIENCE     = 7        # Early stopping

# ImageNet normalization (wajib untuk EfficientNet pretrained)
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

# Output
OUTPUT_DIR = Path("./artifacts")
OUTPUT_DIR.mkdir(exist_ok=True)

print(f"ROOT      : {ROOT}")
print(f"Train dir : {TRAIN_DIR}  {'✅' if os.path.isdir(TRAIN_DIR) else '❌'}")
print(f"Test dir  : {TEST_DIR}   {'✅' if os.path.isdir(TEST_DIR)  else '❌'}")
print(f"Val dir   : {VAL_DIR}    {'✅' if os.path.isdir(VAL_DIR)   else '❌'}")
print(f"IMG_SIZE  : {IMG_SIZE}x{IMG_SIZE}")
print(f"BATCH     : {BATCH_SIZE}")
print(f"EPOCHS    : {NUM_EPOCHS} (max, early stop={PATIENCE})")
print(f"Output    : {OUTPUT_DIR}")

## 🧹 4. Deteksi & Hapus Gambar Rusak
> **NOTE PDF:** *Pastikan melakukan penanganan gambar yang rusak agar proses training berjalan benar.*

In [ ]:
def scan_and_remove_corrupted(root_dir: str, dry_run: bool = False) -> dict:
    """
    Scan seluruh folder, deteksi gambar rusak, dan hapus jika dry_run=False.
    """
    valid_ext   = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
    stats       = {"scanned": 0, "valid": 0, "corrupted": 0, "removed": 0}
    bad_files   = []

    for fp in Path(root_dir).rglob("*"):
        if not fp.is_file() or fp.suffix.lower() not in valid_ext:
            continue
        stats["scanned"] += 1
        try:
            with Image.open(fp) as img:
                img.verify()
            with Image.open(fp) as img:
                img.convert("RGB")
            stats["valid"] += 1
        except Exception as e:
            stats["corrupted"] += 1
            bad_files.append(str(fp))
            if not dry_run:
                fp.unlink()
                stats["removed"] += 1
                print(f"  [HAPUS] {fp.name}  ({type(e).__name__})")

    return stats, bad_files


print("🔍 Scanning gambar rusak di semua folder...")
for folder_name, folder_path in [("train", TRAIN_DIR), ("test", TEST_DIR), ("val", VAL_DIR)]:
    if not os.path.isdir(folder_path):
        print(f"  ⚠️  Folder {folder_name} tidak ada, dilewati.")
        continue
    st, bad = scan_and_remove_corrupted(folder_path, dry_run=False)
    print(f"  📁 {folder_name:5s} → scanned={st['scanned']:4d}  "
          f"valid={st['valid']:4d}  corrupted={st['corrupted']:3d}  "
          f"removed={st['removed']:3d}")

print("\n✅ Scan selesai.")

## 🔄 5. Data Transforms & Augmentasi

In [ ]:
# Training: augmentasi agresif
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE + 32, IMG_SIZE + 32)),
    transforms.RandomCrop(IMG_SIZE),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.15),
    transforms.RandomRotation(degrees=25),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.08),
    transforms.RandomGrayscale(p=0.05),
    transforms.RandomPerspective(distortion_scale=0.2, p=0.3),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

# Validation/Test: tanpa augmentasi
eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

print("✅ Train transform  : Resize→RandomCrop→Flip→Rotate→ColorJitter→Normalize")
print("✅ Eval  transform  : Resize→Normalize")

## 📂 6. Load Dataset

In [ ]:
# ============================================================
# Load dataset dari folder train / val / test
# Dataset ini sudah tersplit → langsung gunakan
# ============================================================

train_dataset = datasets.ImageFolder(root=TRAIN_DIR, transform=train_transform)
val_dataset   = datasets.ImageFolder(root=VAL_DIR,   transform=eval_transform)
test_dataset  = datasets.ImageFolder(root=TEST_DIR,  transform=eval_transform)

CLASS_NAMES = train_dataset.classes
NUM_CLASSES = len(CLASS_NAMES)

print(f"Jumlah kelas  : {NUM_CLASSES}")
print(f"Kelas         : {CLASS_NAMES}")
print(f"Train samples : {len(train_dataset)}")
print(f"Val samples   : {len(val_dataset)}")
print(f"Test samples  : {len(test_dataset)}")
print(f"Total         : {len(train_dataset)+len(val_dataset)+len(test_dataset)}")

## 🔧 7. DataLoaders

In [ ]:
NUM_WORKERS = 2  # Kaggle: bisa naik ke 4

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=True, drop_last=True,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True,
)
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True,
)

print(f"Train batches : {len(train_loader)}")
print(f"Val batches   : {len(val_loader)}")
print(f"Test batches  : {len(test_loader)}")

## 🖼️ 8. Visualisasi Sampel Dataset

In [ ]:
def denorm(t, mean=IMAGENET_MEAN, std=IMAGENET_STD):
    m = torch.tensor(mean).view(3,1,1)
    s = torch.tensor(std).view(3,1,1)
    return (t * s + m).clamp(0,1)

images, labels = next(iter(train_loader))
n = min(8, len(images))
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
axes = axes.flatten()
for i in range(n):
    img = denorm(images[i]).permute(1,2,0).numpy()
    axes[i].imshow(img)
    axes[i].set_title(CLASS_NAMES[labels[i].item()], fontsize=9, pad=3)
    axes[i].axis("off")
for j in range(n, len(axes)):
    axes[j].axis("off")

plt.suptitle("Sampel Data Training (dengan Augmentasi)", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "sample_dataset.png", dpi=120, bbox_inches="tight")
plt.show()
print("✅ Disimpan: artifacts/sample_dataset.png")

## 🧠 9. Definisi Model — EfficientNet-B3 (Transfer Learning)

In [ ]:
def build_model(num_classes: int) -> nn.Module:
    """
    EfficientNet-B3 pretrained ImageNet.
    Classifier head diganti sesuai jumlah kelas dataset.
    Semua layer dilatih (fine-tune penuh).
    """
    model = timm.create_model(
        "efficientnet_b3",
        pretrained=True,
        num_classes=num_classes,
        drop_rate=0.3,
        drop_path_rate=0.2,
    )
    return model


model     = build_model(NUM_CLASSES).to(DEVICE)
total_p   = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Arsitektur       : EfficientNet-B3 (timm)")
print(f"Total params     : {total_p:,}")
print(f"Trainable params : {trainable:,}")
print(f"Classifier head  : {model.classifier}")

## 📉 10. Loss, Optimizer & Scheduler

In [ ]:
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

optimizer = optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY,
)

scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=NUM_EPOCHS, eta_min=1e-6,
)

print("Loss      : CrossEntropyLoss (label_smoothing=0.1)")
print("Optimizer : AdamW (lr=3e-4, wd=1e-4)")
print("Scheduler : CosineAnnealingLR")

## ⏱️ 11. Early Stopping & Training Loop

In [ ]:
class EarlyStopping:
    """
    Hentikan training jika val_loss tidak membaik selama `patience` epoch.
    Pulihkan bobot terbaik secara otomatis.
    """
    def __init__(self, patience: int = 7, min_delta: float = 1e-4):
        self.patience    = patience
        self.min_delta   = min_delta
        self.best_loss   = float("inf")
        self.best_acc    = 0.0
        self.best_epoch  = 0
        self.best_state  = None
        self.counter     = 0
        self.stopped     = False

    def step(self, val_loss, val_acc, model, epoch):
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss  = val_loss
            self.best_acc   = val_acc
            self.best_epoch = epoch
            self.best_state = {k: v.clone().cpu() for k, v in model.state_dict().items()}
            self.counter    = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.stopped = True
                model.load_state_dict(self.best_state)
                print(f"  ⏹  Early stopping! "
                      f"best_epoch={self.best_epoch}, "
                      f"val_loss={self.best_loss:.4f}, "
                      f"val_acc={self.best_acc*100:.2f}%")
        return self.stopped


def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for imgs, labels in tqdm(loader, desc="  [Train]", leave=False):
        imgs, labels = imgs.to(device, non_blocking=True), labels.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        out  = model(imgs)
        loss = criterion(out, labels)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item() * imgs.size(0)
        correct    += out.argmax(1).eq(labels).sum().item()
        total      += imgs.size(0)
    return total_loss / total, correct / total


@torch.no_grad()
def eval_one_epoch(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    for imgs, labels in tqdm(loader, desc="  [Val]  ", leave=False):
        imgs, labels = imgs.to(device, non_blocking=True), labels.to(device, non_blocking=True)
        out  = model(imgs)
        loss = criterion(out, labels)
        total_loss += loss.item() * imgs.size(0)
        correct    += out.argmax(1).eq(labels).sum().item()
        total      += imgs.size(0)
    return total_loss / total, correct / total


print("✅ EarlyStopping + fungsi training/evaluasi siap!")

## 🚀 12. Jalankan Training

In [ ]:
history = {"train_loss":[], "val_loss":[], "train_acc":[], "val_acc":[], "lr":[]}
early_stopper = EarlyStopping(patience=PATIENCE)

print("=" * 68)
print(f"  🚀  Training EfficientNet-B3 | {NUM_CLASSES} kelas | device={DEVICE}")
print(f"  Max epoch: {NUM_EPOCHS}  |  Early stopping patience: {PATIENCE}")
print("=" * 68)

for epoch in range(1, NUM_EPOCHS + 1):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, DEVICE)
    val_loss,   val_acc   = eval_one_epoch(model, val_loader, criterion, DEVICE)
    lr_now = optimizer.param_groups[0]["lr"]
    scheduler.step()

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["train_acc"].append(train_acc)
    history["val_acc"].append(val_acc)
    history["lr"].append(lr_now)

    star = " ⭐" if val_loss <= early_stopper.best_loss else ""
    print(
        f"Epoch [{epoch:02d}/{NUM_EPOCHS}]  "
        f"Train  loss={train_loss:.4f}  acc={train_acc*100:5.2f}%  |  "
        f"Val  loss={val_loss:.4f}  acc={val_acc*100:5.2f}%  "
        f"lr={lr_now:.2e}{star}"
    )

    if early_stopper.step(val_loss, val_acc, model, epoch):
        break

print(f"\n✅ Training selesai!")
print(f"   Best epoch    : {early_stopper.best_epoch}")
print(f"   Best val loss : {early_stopper.best_loss:.4f}")
print(f"   Best val acc  : {early_stopper.best_acc*100:.2f}%")

## 📈 13. Training & Validation Curve

In [ ]:
ep = range(1, len(history["train_loss"]) + 1)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("Training & Validation Curves — EfficientNet-B3",
             fontsize=14, fontweight="bold")

# Loss
axes[0].plot(ep, history["train_loss"], "b-o", ms=3, label="Train Loss")
axes[0].plot(ep, history["val_loss"],   "r-o", ms=3, label="Val Loss")
axes[0].axvline(early_stopper.best_epoch, color="green", ls="--", alpha=0.7,
                label=f"Best ({early_stopper.best_epoch})")
axes[0].set_title("Loss", fontweight="bold")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss")
axes[0].legend(); axes[0].grid(True, alpha=0.3)

# Accuracy
ta = [x*100 for x in history["train_acc"]]
va = [x*100 for x in history["val_acc"]]
axes[1].plot(ep, ta, "b-o", ms=3, label="Train Acc")
axes[1].plot(ep, va, "r-o", ms=3, label="Val Acc")
axes[1].axvline(early_stopper.best_epoch, color="green", ls="--", alpha=0.7)
axes[1].set_title("Accuracy", fontweight="bold")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Accuracy (%)")
axes[1].legend(); axes[1].grid(True, alpha=0.3)

# Learning Rate
axes[2].plot(ep, history["lr"], "g-o", ms=3)
axes[2].set_title("Learning Rate Schedule", fontweight="bold")
axes[2].set_xlabel("Epoch"); axes[2].set_ylabel("LR")
axes[2].set_yscale("log"); axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "training_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Disimpan: artifacts/training_curves.png")

## 📊 14. Evaluasi Model — Accuracy, Classification Report & Confusion Matrix

In [ ]:
@torch.no_grad()
def collect_preds(model, loader, device):
    model.eval()
    all_preds, all_labels = [], []
    for imgs, labels in tqdm(loader, desc="Evaluating"):
        imgs   = imgs.to(device, non_blocking=True)
        preds  = model(imgs).argmax(1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())
    return np.array(all_labels), np.array(all_preds)


print("Mengevaluasi pada TEST set...")
y_true, y_pred = collect_preds(model, test_loader, DEVICE)

acc = accuracy_score(y_true, y_pred)
print(f"\n🎯 Test Accuracy : {acc*100:.2f}%\n")
print("📋 Classification Report:")
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, digits=4))

# Simpan ke file
report_str = classification_report(y_true, y_pred, target_names=CLASS_NAMES, digits=4)
with open(OUTPUT_DIR / "classification_report.txt", "w") as f:
    f.write(f"Test Accuracy: {acc*100:.2f}%\n\n")
    f.write(report_str)
print("✅ Disimpan: artifacts/classification_report.txt")

In [ ]:
# Confusion Matrix
cm      = confusion_matrix(y_true, y_pred)
cm_norm = cm.astype("float") / cm.sum(axis=1)[:, np.newaxis]

fsz = max(10, NUM_CLASSES + 2)
plt.figure(figsize=(fsz, fsz - 1))
sns.heatmap(
    cm_norm, annot=cm, fmt="d",
    cmap="YlOrRd",
    xticklabels=CLASS_NAMES,
    yticklabels=CLASS_NAMES,
    linewidths=0.4, square=True,
    cbar_kws={"label": "Proporsi"},
)
plt.title(f"Confusion Matrix — Test Set  (Acc: {acc*100:.2f}%)",
          fontsize=13, fontweight="bold", pad=12)
plt.xlabel("Predicted Label", fontsize=11, labelpad=8)
plt.ylabel("True Label",      fontsize=11, labelpad=8)
plt.xticks(rotation=45, ha="right", fontsize=9)
plt.yticks(rotation=0, fontsize=9)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Disimpan: artifacts/confusion_matrix.png")

## 💾 15. Export Model (TorchScript) & Metadata

In [ ]:
model.eval()
model_cpu = model.cpu()

model_path = OUTPUT_DIR / "insect_model.pt"
try:
    scripted = torch.jit.script(model_cpu)
    torch.jit.save(scripted, str(model_path))
    export_fmt = "torchscript"
    print(f"✅ TorchScript model disimpan : {model_path}")
except Exception as e:
    print(f"⚠️  jit.script gagal ({e})\n   Fallback ke state_dict...")
    torch.save({"model_state_dict": model_cpu.state_dict(),
                "model_arch": "efficientnet_b3"}, str(model_path))
    export_fmt = "state_dict"
    print(f"✅ State dict disimpan        : {model_path}")

model = model.to(DEVICE)   # kembalikan ke device

# Metadata — dibaca oleh ml_service.py di backend
metadata = {
    "model_arch"   : "efficientnet_b3",
    "class_names"  : CLASS_NAMES,
    "num_classes"  : NUM_CLASSES,
    "img_size"     : IMG_SIZE,
    "imagenet_mean": IMAGENET_MEAN,
    "imagenet_std" : IMAGENET_STD,
    "best_val_acc" : float(early_stopper.best_acc),
    "best_val_loss": float(early_stopper.best_loss),
    "best_epoch"   : int(early_stopper.best_epoch),
    "test_accuracy": float(acc),
    "total_epochs" : len(history["train_loss"]),
    "export_format": export_fmt,
}

meta_path = OUTPUT_DIR / "metadata.json"
with open(meta_path, "w") as f:
    json.dump(metadata, f, indent=2)

print(f"✅ Metadata disimpan          : {meta_path}")
print(f"\n📦 File yang perlu di-download:")
print(f"   1. {model_path}  (→ backend/artifacts/)")
print(f"   2. {meta_path}   (→ backend/artifacts/)")

## 🔍 16. Verifikasi Model yang Disimpan

In [ ]:
print("Verifikasi model TorchScript...")

loaded = torch.jit.load(str(model_path), map_location="cpu")
loaded.eval()

dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE)
with torch.no_grad():
    out = loaded(dummy)

assert out.shape == (1, NUM_CLASSES), (
    f"Output shape salah: {out.shape}, expected (1, {NUM_CLASSES})"
)
print(f"  Input shape  : {dummy.shape}")
print(f"  Output shape : {out.shape}")
print(f"  Num classes  : {out.shape[1]}")
print(f"\n✅ Model verified! Siap untuk deployment di backend.")
print(f"\n📋 Ringkasan Training:")
print(f"   Arsitektur   : EfficientNet-B3")
print(f"   Total epoch  : {len(history['train_loss'])}")
print(f"   Best epoch   : {early_stopper.best_epoch}")
print(f"   Val accuracy : {early_stopper.best_acc*100:.2f}%")
print(f"   Test accuracy: {acc*100:.2f}%")
print(f"   Kelas        : {CLASS_NAMES}")